# Galassi 2012 - Gate C: Fill-Rate Trend & Locked Parameters

This notebook implements Gate C. It requires Gate B to have passed. It explores the sensitivity of the peak temperature to fill time (fill rate) and exports the locked model parameters.

In [ ]:
import os
import sys
import json
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add repo to path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/H2_Storage'
except ImportError:
    BASE_DIR = 'H2_Storage'
    print(f"Using local BASE_DIR: {BASE_DIR}")

REPO_DIR = os.path.join(BASE_DIR, 'repo')
sys.path.append(REPO_DIR)

try:
    from h2tank.galassi_baseline import simulate_fast_fill
except ImportError:
    # Attempt relative import
    sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../repo')))
    from h2tank.galassi_baseline import simulate_fast_fill

DATA_DIR = os.path.join(BASE_DIR, 'validation/data/galassi_2012')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
FIGURES_DIR = os.path.join(BASE_DIR, 'figures')
CHECKPOINTS_DIR = os.path.join(BASE_DIR, 'checkpoints')

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)

## A) Check Gate B Status

In [ ]:
metrics_path = os.path.join(RESULTS_DIR, 'galassi2012_gateB_metrics.csv')
if not os.path.exists(metrics_path):
    raise FileNotFoundError("Gate B metrics file not found. Run Gate B first.")

df_metrics = pd.read_csv(metrics_path)

gate_b_passed = False

if 'Status' in df_metrics.columns and (df_metrics['Status'] == 'INCOMPLETE').any():
    print("Gate B Status: INCOMPLETE")
    gate_b_passed = False
elif 'Passed' in df_metrics.columns:
    if df_metrics['Passed'].all():
        print("Gate B Status: PASSED")
        gate_b_passed = True
    else:
        print("Gate B Status: FAILED")
        gate_b_passed = False
else:
    print("Gate B Status: UNKNOWN (Invalid CSV format)")
    gate_b_passed = False

if not gate_b_passed:
    print("STOP: Gate B must pass before Gate C.")
    # Stop execution here in a real notebook
    # raising an exception to halt automated runs
    raise RuntimeError("Gate B must pass before Gate C.")

## B & C) Load Protocol and Locked Parameters

In [ ]:
# Load Protocol
protocol_path = os.path.join(DATA_DIR, 'protocol_table1.json')
with open(protocol_path, 'r') as f:
    protocols = json.load(f)
    
# Use H2_250 as baseline
baseline_protocol = protocols['H2_250']

# Load Gate B Checkpoint to get locked UA
checkpoint_b_path = os.path.join(CHECKPOINTS_DIR, 'galassi2012_gateB.pkl')
with open(checkpoint_b_path, 'rb') as f:
    gate_b_data = pickle.load(f)
    
ua_locked = gate_b_data['model_params']['UA']
print(f"Locked UA from Gate B: {ua_locked} W/K")

# Define locked parameters for Gate C
locked_params = {
    'UA': ua_locked,
    'vol_m3': 0.029,  # Enforce 29L as per requirements
    'mass_tank_kg': 0.0, # Assuming ignored as per baseline
    'cp_tank': 0.0
}
print("Locked Parameters:", locked_params)

## D) Fill Time Sweep

In [ ]:
t_fill_sweep = [180, 245, 330, 450, 600]
results_sweep = []

print("Starting sweep...")
for t_fill in t_fill_sweep:
    # Copy baseline protocol and update t_fill
    current_protocol = baseline_protocol.copy()
    current_protocol['t_fill_s'] = t_fill
    
    # Run simulation
    sim_res = simulate_fast_fill(current_protocol, locked_params)
    
    # Metrics
    T_peak = sim_res['T_peak_K']
    Tamb_K = current_protocol['Tamb_C'] + 273.15
    DeltaT_peak = T_peak - Tamb_K
    
    # Fill rate proxy (bar/s)
    P_in = current_protocol['Pin_bar']
    P_fin = current_protocol['Pfin_bar']
    fillrate_proxy = (P_fin - P_in) / t_fill
    
    results_sweep.append({
        't_fill_s': t_fill,
        'fillrate_bar_s': fillrate_proxy,
        'T_peak_K': T_peak,
        'DeltaT_peak_K': DeltaT_peak
    })
    print(f"  t_fill={t_fill}s, Rate={fillrate_proxy:.2f} bar/s -> DeltaT={DeltaT_peak:.2f} K")

df_sweep = pd.DataFrame(results_sweep)

## E) Plotting

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(df_sweep['fillrate_bar_s'], df_sweep['DeltaT_peak_K'], 'o-')
plt.xlabel('Fill Rate Proxy (bar/s)')
plt.ylabel('Peak Temperature Rise (K)')
plt.title('Galassi 2012: Fill Rate Trend (Gate C)')
plt.grid(True)

fig_path = os.path.join(FIGURES_DIR, 'galassi2012_fillrate_trend.png')
plt.savefig(fig_path)
print(f"Saved plot to {fig_path}")
plt.close()

## F, G, H) Metrics, Locked Params, and Trend Check

In [ ]:
# F) Save Metrics Table
csv_path = os.path.join(RESULTS_DIR, 'galassi2012_gateC_metrics.csv')
df_sweep.to_csv(csv_path, index=False)
print(f"Saved metrics to {csv_path}")

# G) Save Locked Params
json_path = os.path.join(RESULTS_DIR, 'galassi2012_locked_params.json')
with open(json_path, 'w') as f:
    json.dump(locked_params, f, indent=4)
print(f"Saved locked params to {json_path}")

# H) Trend Check
# Assert DeltaT_peak increases as fillrate_proxy increases
# Sort by fillrate just in case
df_sorted = df_sweep.sort_values('fillrate_bar_s')
deltas = df_sorted['DeltaT_peak_K'].values

# Check monotonicity
# Allow minor noise? The prompt says "allow minor tolerance".
# But physically it should be strictly monotonic for this simple model.
is_monotonic = np.all(np.diff(deltas) >= -0.5) # Tolerance of 0.5K for noise

if not is_monotonic:
    print("WARNING: Trend check failed! DeltaT did not strictly increase with fill rate.")
    # Flag in metrics? We already saved the CSV. Overwrite or append status?
    # Prompt says "flag 'TREND_CHECK_FAILED' in metrics".
    # We can add a metadata file or append to the CSV logic.
    # Let's add a status file or print to stdout.
    with open(os.path.join(RESULTS_DIR, 'galassi2012_gateC_status.txt'), 'w') as f:
        f.write("TREND_CHECK_FAILED")
else:
    print("Trend check PASSED.")
    with open(os.path.join(RESULTS_DIR, 'galassi2012_gateC_status.txt'), 'w') as f:
        f.write("PASSED")

## I) Save Checkpoint

In [ ]:
checkpoint_c_data = {
    'locked_params': locked_params,
    'sweep_results': df_sweep.to_dict('records'),
    'trend_check_passed': bool(is_monotonic)
}

checkpoint_c_path = os.path.join(CHECKPOINTS_DIR, 'galassi2012_gateC.pkl')
with open(checkpoint_c_path, 'wb') as f:
    pickle.dump(checkpoint_c_data, f)
print(f"Saved checkpoint to {checkpoint_c_path}")